# CatBoost2 — Entrenamiento Cloud (Kaggle Notebooks + GPU)

**Proyecto:** MECMT07 — Home Credit Default Risk  
**Versión:** v2 — 5-fold CV + variables categóricas nativas  
**Plataforma:** Kaggle Notebooks con GPU P100 gratuita  
**Dataset fuente:** `mecmt07-features` (Dataset privado de Kaggle)

**Diferencias respecto a catboost-cloud-train.ipynb (v1):**
- **CV 5-fold** con `catboost.cv()` en lugar de split 80/20 → evaluación comparable con XGBoost y LightGBM
- **Variables categóricas nativas**: lee `features_train_cat.parquet` que incluye 6 columnas categóricas (NAME_CONTRACT_TYPE, NAME_INCOME_TYPE, NAME_EDUCATION_TYPE, NAME_FAMILY_STATUS, NAME_HOUSING_TYPE, OCCUPATION_TYPE). CatBoost las procesa con target encoding ordenado sin data leakage.

**Estrategia:**
- CatBoost maneja NaN y categóricas nativamente
- `auto_class_weights='Balanced'` para desbalance de clases
- Optuna 20 trials: objetivo = CV AUC − 0.5 · gap (penaliza overfitting)
- `catboost.cv()` con 5-fold estratificado y early stopping
- GPU P100 con `task_type='GPU'`

**Outputs en `/kaggle/working/`:**
- `catboost2_cloud_best.cbm` — modelo final (formato binario CatBoost)
- `catboost2_cloud_metadata.json` — hiperparámetros, AUC, feature_cols, cat_cols
- `optuna_trials.csv` — historial de todos los trials

**Flujo post-entrenamiento:**
1. Descargar `.cbm`, `.json`, `.csv` desde el Output tab
2. Subir `features_test_cat.parquet` al dataset de Kaggle
3. Correr localmente `catboost2_cloud_predict.ipynb`

In [ ]:
# ─── Instalar dependencias ────────────────────────────────────────────────────
!pip install optuna --quiet
print('Dependencias listas.')

In [ ]:
# ─── Verificar GPU ────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    USE_GPU = True
else:
    print('GPU no disponible — se usará CPU.')
    USE_GPU = False
print(f'USE_GPU = {USE_GPU}')

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import catboost
from catboost import CatBoostClassifier, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

print(f'CatBoost version : {catboost.__version__}')
print(f'Optuna  version  : {optuna.__version__}')
print('Imports OK')

In [ ]:
# ─── Configuración ────────────────────────────────────────────────────────────
DATA_DIR  = Path('/kaggle/input/datasets/davidguzzi/mecmt07-features')
MODEL_DIR = Path('/kaggle/working')

N_TRIALS   = 20
N_FOLDS    = 5
EARLY_STOP = 50
MAX_ITER   = 1000
SEED       = 42

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 12,
    'axes.spines.top': False, 'axes.spines.right': False
})

print('=' * 60)
print('CONFIGURACIÓN')
print(f'  DATA_DIR   : {DATA_DIR}')
print(f'  MODEL_DIR  : {MODEL_DIR}')
print(f'  N_TRIALS   : {N_TRIALS}')
print(f'  N_FOLDS    : {N_FOLDS}  (CV 5-fold)')
print(f'  EARLY_STOP : {EARLY_STOP}')
print(f'  MAX_ITER   : {MAX_ITER}')
print(f'  SEED       : {SEED}')
print(f'  GPU        : {USE_GPU}')
print('=' * 60)

In [ ]:
# ─── Cargar datos ─────────────────────────────────────────────────────────────
# Lee features_train_cat.parquet (30 features numéricas + 6 categóricas)
print('Cargando datos...')
df      = pd.read_parquet(DATA_DIR / 'features_train_cat.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test_cat.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

# Detectar columnas categóricas (object dtype) → CatBoost las procesa nativamente
cat_cols         = [c for c in feature_cols if df[c].dtype == 'object']
cat_features_idx = [feature_cols.index(c) for c in cat_cols]

# X como DataFrame (preserva dtypes para CatBoost)
X           = df[feature_cols]
y           = df['TARGET'].values
X_test      = df_test[feature_cols]
sk_ids_test = df_test['SK_ID_CURR'].values

n_neg, n_pos = (y == 0).sum(), (y == 1).sum()

# Pool para catboost.cv()
pool = Pool(X, label=y, cat_features=cat_features_idx, feature_names=feature_cols)

print(f'  Train shape        : {X.shape}')
print(f'  Test  shape        : {X_test.shape}')
print(f'  Features totales   : {len(feature_cols)}')
print(f'  Features numéricas : {len(feature_cols) - len(cat_cols)}')
print(f'  Features categóric.: {len(cat_cols)}  → {cat_cols}')
print(f'  TARGET=0 (paga)    : {n_neg:,}  ({100*n_neg/(n_neg+n_pos):.1f}%)')
print(f'  TARGET=1 (default) : {n_pos:,}  ({100*n_pos/(n_neg+n_pos):.1f}%)')

In [ ]:
# ─── Funciones auxiliares ─────────────────────────────────────────────────────
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

print('Funciones auxiliares definidas.')

## 1. Optuna — Búsqueda de Hiperparámetros

**Función objetivo** con `catboost.cv()` + 5-fold estratificado:
$$\text{objetivo} = \text{AUC}_{\text{cv}} - 0.5 \cdot \max(0,\ \text{AUC}_{\text{train}} - \text{AUC}_{\text{cv}})$$

In [ ]:
# ─── Optuna objective con catboost.cv() ───────────────────────────────────────
best_iter_per_trial = {}

def objective(trial):
    _bt_options = ['Bayesian', 'Bernoulli'] if USE_GPU else ['Bayesian', 'Bernoulli', 'MVS']
    bootstrap_type = trial.suggest_categorical('bootstrap_type', _bt_options)

    params = dict(
        iterations            = MAX_ITER,
        learning_rate         = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        depth                 = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg           = trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        bootstrap_type        = bootstrap_type,
        random_strength       = trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        auto_class_weights    = 'Balanced',
        eval_metric           = 'AUC',
        early_stopping_rounds = EARLY_STOP,
        task_type             = 'GPU' if USE_GPU else 'CPU',
        random_seed           = SEED,
        verbose               = False,
    )

    if bootstrap_type == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 10.0)
    else:
        params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)

    # catboost.cv() — equivalente a lgb.cv(); 5-fold estratificado
    cv_result = catboost.cv(
        pool,
        params,
        fold_count            = N_FOLDS,
        stratified            = True,
        early_stopping_rounds = EARLY_STOP,
        verbose               = False,
        as_pandas             = True,
    )

    best_idx  = cv_result['test-AUC-mean'].idxmax()
    cv_auc    = cv_result.loc[best_idx, 'test-AUC-mean']
    train_auc = cv_result.loc[best_idx, 'train-AUC-mean']
    n_iter    = best_idx + 1

    best_iter_per_trial[trial.number] = n_iter
    trial.set_user_attr('train_auc', train_auc)
    trial.set_user_attr('cv_auc',    cv_auc)
    trial.set_user_attr('n_iter',    n_iter)

    gap = max(0.0, train_auc - cv_auc)
    return cv_auc - 0.5 * gap

print('Función objetivo definida.')

In [ ]:
# ─── Optuna study — 20 trials ─────────────────────────────────────────────────
def _optuna_callback(study, trial):
    n      = best_iter_per_trial.get(trial.number, '?')
    marker = ' ◀ best' if trial.number == study.best_trial.number else ''
    print(f'  Trial {trial.number + 1:>2}/{N_TRIALS} │ '
          f'obj={trial.value:.5f}  '
          f'iter={str(n):>4}  │  '
          f'best={study.best_value:.5f}{marker}')

print('=' * 65)
print(f'Lanzando Optuna — {N_TRIALS} trials')
print(f'CV: {N_FOLDS}-fold estratificado  |  Early stopping: {EARLY_STOP} rounds')
print(f'Cat features: {cat_cols}')
print(f'Device: {"GPU" if USE_GPU else "CPU"}')
print('=' * 65)

study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(
    objective,
    n_trials          = N_TRIALS,
    n_jobs            = 1,
    show_progress_bar = False,
    callbacks         = [_optuna_callback]
)

best_n_iter = best_iter_per_trial[study.best_trial.number]

print('=' * 65)
print(f'Búsqueda finalizada.')
print(f'  Mejor CV AUC (obj penalizado) : {study.best_value:.5f}')
print(f'  CV AUC real                   : {study.best_trial.user_attrs["cv_auc"]:.5f}')
print(f'  Train AUC                     : {study.best_trial.user_attrs["train_auc"]:.5f}')
print(f'  n_iterations óptimas          : {best_n_iter}')
print(f'  Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'    {k:<24}: {v}')
print('=' * 65)

trial_df = study.trials_dataframe()
trial_df.to_csv(MODEL_DIR / 'optuna_trials.csv', index=False)
print(f'\nHistorial de trials guardado.')

In [ ]:
# ─── Historial de Optuna ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trial_df['number'], trial_df['value'], 'o-', color='#e67e22', ms=6)
ax.axhline(study.best_value, color='#3498db', ls='--',
           label=f'Best = {study.best_value:.5f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Objetivo (CV AUC penalizado)')
ax.set_title('Optuna — CatBoost2 Cloud (5-fold CV)')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'optuna_history.png', dpi=120)
plt.show()

## 2. Modelo Final — Refit en Train Completo

In [ ]:
# ─── Entrenar modelo final ────────────────────────────────────────────────────
params_final = dict(
    iterations         = best_n_iter,
    auto_class_weights = 'Balanced',
    eval_metric        = 'AUC',
    task_type          = 'GPU' if USE_GPU else 'CPU',
    random_seed        = SEED,
    verbose            = 100,
    **study.best_params
)

print('=' * 65)
print('Entrenando modelo final en train completo...')
print(f'  Iteraciones : {best_n_iter}')
print(f'  Cat features: {cat_cols}')
print(f'  Device      : {"GPU" if USE_GPU else "CPU"}')
print('=' * 65)

t0 = time.time()
model = CatBoostClassifier(**params_final)
model.fit(X, y, cat_features=cat_features_idx)
elapsed = time.time() - t0
print(f'Entrenamiento finalizado en {elapsed:.0f}s ✓')

## 3. Métricas sobre Train Completo

In [ ]:
# ─── Métricas finales ─────────────────────────────────────────────────────────
y_prob_train = model.predict_proba(X)[:, 1]
metrics      = compute_metrics(y, y_prob_train, threshold=0.5, label='CatBoost2 Cloud')
metrics['CV_AUC']  = round(study.best_trial.user_attrs['cv_auc'], 5)
metrics['n_iter']  = best_n_iter

print('=' * 65)
print('CATBOOST2 CLOUD — MÉTRICAS FINALES')
print('=' * 65)
print(f'  Train AUC         : {metrics["AUC"]}')
print(f'  CV AUC (5-fold)   : {metrics["CV_AUC"]}  (comparable con XGB y LGB)')
print(f'  Obj penalizado    : {study.best_value:.5f}')
print(f'  Gap Train - CV    : {metrics["AUC"] - metrics["CV_AUC"]:.5f}')
print(f'  n_iterations      : {metrics["n_iter"]}')
print(f'  Recall            : {metrics["Recall"]}')
print(f'  Precision         : {metrics["Precision"]}')
print(f'  F1                : {metrics["F1"]}')
print(f'  TP/TN/FP/FN       : {metrics["TP"]} / {metrics["TN"]} / {metrics["FP"]} / {metrics["FN"]}')
print('=' * 65)

display(pd.DataFrame([metrics]).set_index('Model'))

In [ ]:
# ─── Curva ROC ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y, y_prob_train)
auc_val      = roc_auc_score(y, y_prob_train)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#e67e22', lw=2, label=f'CatBoost2 Cloud (AUC = {auc_val:.5f})')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — CatBoost2 Cloud (train)')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'roc_curve.png', dpi=120)
plt.show()

## 4. Importancia de Variables

In [ ]:
# ─── Feature importance ───────────────────────────────────────────────────────
importance  = model.get_feature_importance()
feat_imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importance})
feat_imp_df['is_cat'] = feat_imp_df['feature'].isin(cat_cols)
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)

top15 = feat_imp_df.head(15)
colors = ['#c0392b' if is_cat else '#e67e22' for is_cat in top15['is_cat'][::-1]]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top15['feature'][::-1], top15['importance'][::-1],
               color=colors, edgecolor='white')
ax.set_xlabel('Importancia')
ax.set_title('Top 15 Features — CatBoost2 Cloud\n(rojo = categóricas nativas, naranja = numéricas)')
for bar, val in zip(bars, top15['importance'][::-1]):
    ax.text(bar.get_width() + top15['importance'].max()*0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'feature_importance.png', dpi=120)
plt.show()
display(feat_imp_df.reset_index(drop=True))

In [ ]:
# ─── Distribución de scores ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_prob_train[y == 0], bins=60, alpha=0.6, color='#3498db',
        label='TARGET=0 (paga)', density=True)
ax.hist(y_prob_train[y == 1], bins=60, alpha=0.6, color='#e74c3c',
        label='TARGET=1 (default)', density=True)
ax.set_xlabel('Probabilidad predicha P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — CatBoost2 Cloud')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'score_distribution.png', dpi=120)
plt.show()

## 5. Guardar Modelo y Metadata

In [ ]:
# ─── Guardar modelo ───────────────────────────────────────────────────────────
model_path = MODEL_DIR / 'catboost2_cloud_best.cbm'
model.save_model(str(model_path))

metadata = {
    'best_params'      : {k: float(v) if isinstance(v, (int, float)) else v
                          for k, v in study.best_params.items()},
    'best_n_rounds'    : int(best_n_iter),
    'best_n_iter'      : int(best_n_iter),
    'best_cv_auc'      : float(study.best_trial.user_attrs['cv_auc']),
    'best_obj_auc'     : float(study.best_value),
    'train_auc'        : float(metrics['AUC']),
    'feature_cols'     : list(feature_cols),
    'cat_cols'         : list(cat_cols),
    'cat_features_idx' : list(cat_features_idx),
    'n_trials'         : N_TRIALS,
    'n_folds'          : N_FOLDS,
    'method'           : 'cv',
    'early_stop'       : EARLY_STOP,
    'use_gpu'          : USE_GPU,
    'catboost_version' : catboost.__version__,
    'timestamp'        : pd.Timestamp.now().isoformat()
}

meta_path = MODEL_DIR / 'catboost2_cloud_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('=' * 65)
print('ARTEFACTOS GUARDADOS')
print('=' * 65)
print(f'  {model_path.name:<40} ({model_path.stat().st_size/1e6:.2f} MB)')
print(f'  {meta_path.name}')
print(f'  optuna_trials.csv')
print('=' * 65)
print('\n>>> Descargar desde el Output tab de Kaggle:')
print(f'    - {model_path.name}')
print(f'    - {meta_path.name}')
print(f'    - optuna_trials.csv')
print('\n>>> Luego correr localmente: catboost2_cloud_predict.ipynb')

## Resumen Final

In [ ]:
# ─── Resumen ──────────────────────────────────────────────────────────────────
print('=' * 65)
print('CATBOOST2 CLOUD — RESUMEN')
print('=' * 65)
print('Mejoras respecto a CatBoost v1:')
print('  - Evaluación: 5-fold CV (comparable con XGBoost y LightGBM)')
print(f'  - Variables categóricas nativas: {cat_cols}')
print('\nHiperparámetros óptimos (Optuna):')
for k, v in study.best_params.items():
    print(f'  {k:<24}: {v}')
print(f'  {"n_iterations":<24}: {best_n_iter}  (via early stopping en CV)')
print('\nMétricas:')
print(f'  Train AUC       : {metrics["AUC"]}')
print(f'  CV AUC (5-fold) : {metrics["CV_AUC"]}')
print(f'  Gap Train - CV  : {metrics["AUC"] - metrics["CV_AUC"]:.5f}')
print(f'  Recall          : {metrics["Recall"]}')
print(f'  Precision       : {metrics["Precision"]}')
print(f'  F1              : {metrics["F1"]}')
print('\nEntorno:')
print(f'  CatBoost version: {catboost.__version__}')
print(f'  GPU usado       : {USE_GPU}')
print(f'  Timestamp       : {metadata["timestamp"]}')
print('=' * 65)